# PyHealth Medical Code Ontology Tutorial

This notebook covers **`pyhealth.medcode`** — a medical code ontology library for looking up codes, exploring hierarchies, and translating between code systems.

You will learn:
- How to load a medical code system using **`InnerMap`**
- How to look up diabetes codes in **ICD-10-CM** with detailed explanations
- How to traverse the **code hierarchy** (ancestors and descendants)
- How to **translate codes** between systems using **`CrossMap`** (e.g., ICD-9 → ICD-10, ICD-10 → CCS)
- Practical patterns for preprocessing EHR datasets

---

In [ ]:
from pyhealth.medcode import InnerMap, CrossMap

---
## Background: Medical Code Systems

EHR data uses several overlapping code systems:

| System | Domain | Example |
|--------|--------|----------|
| **ICD-10-CM** | Diagnoses (current US standard) | `E11.9` = Type 2 DM |
| **ICD-9-CM** | Diagnoses (legacy, pre-2015) | `250.00` = Type 2 DM |
| **ICD-10-PCS / ICD-9-PROC** | Procedures | |
| **ATC** | Drug classification hierarchy | `A10BA02` = Metformin |
| **RxNorm** | Drug concepts (US) | `860975` = Metformin 500mg tablet |
| **NDC** | Drug product codes (package level) | |
| **CCSCM** | Clinical Classifications Software — Diagnoses | `49` = Diabetes mellitus |
| **CCSPROC** | CCS — Procedures | |

**Why code mapping matters in ML:**
- MIMIC-III uses ICD-9 codes (pre-2015 data), while MIMIC-IV uses ICD-10
- Vocabularies differ in specificity: ICD-10 has ~70,000 codes vs ICD-9's ~14,000
- ML models trained on ICD-9 codes cannot directly generalize to ICD-10 datasets without mapping
- CCS groups 70,000 ICD-10 codes into ~300 categories — much better for small datasets

---
## Part 1: Loading a Code System

`InnerMap.load(vocabulary)` downloads and caches the code system on first call, then loads from the local cache on subsequent calls.

In [ ]:
# Load ICD-10-CM (Clinical Modification) — the US standard for diagnosis codes
icd10cm = InnerMap.load("ICD10CM")

# Print statistics
icd10cm.stat()

In [ ]:
# What attributes are available for each code?
print("Available attributes:", icd10cm.available_attributes)

In [ ]:
# Check code existence
print("'E11.9' in ICD10CM:", "E11.9" in icd10cm)
print("'E11.99' in ICD10CM:", "E11.99" in icd10cm)   # non-existent code
print("'XYZ' in ICD10CM:",   "XYZ"   in icd10cm)

---
## Part 2: Diabetes Code Deep Dive

Diabetes mellitus is encoded in ICD-10-CM chapter E08–E13. Here's the full taxonomy:

```
E08  Diabetes mellitus due to underlying condition
E09  Drug or chemical induced diabetes mellitus
E10  Type 1 diabetes mellitus
E11  Type 2 diabetes mellitus
E13  Other specified diabetes mellitus
```

Each category expands into complication subtypes. The E11 (Type 2) branch has **87 billable codes** in 2026 ICD-10-CM — a testament to how granular modern clinical coding is:

```
E11    Type 2 DM
├─ E11.00 / E11.01   with hyperosmolarity (without / with coma)
├─ E11.10 / E11.11   with ketoacidosis (without / with coma)
├─ E11.2   with kidney complications
│   ├─ E11.21  with diabetic nephropathy
│   └─ E11.22  with diabetic chronic kidney disease
├─ E11.3   with ophthalmic complications
│   ├─ E11.311 / E11.319  unspecified retinopathy (with / without macular edema)
│   ├─ E11.321x–E11.359x  nonproliferative/proliferative retinopathy
│   │                      (each further split: right eye / left eye / bilateral / unspecified)
│   └─ E11.36   with diabetic cataract
├─ E11.4   with neurological complications
│   ├─ E11.40  with diabetic neuropathy, unspecified
│   └─ E11.42  with diabetic polyneuropathy
├─ E11.5   with circulatory complications
│   ├─ E11.51  with peripheral angiopathy without gangrene
│   └─ E11.52  with peripheral angiopathy with gangrene
├─ E11.6   with other specified complications
│   ├─ E11.65  with hyperglycemia
│   └─ E11.69  with other specified complication
├─ E11.8   with unspecified complications
├─ E11.9   without complications  (most common at initial diagnosis)
└─ E11.A   without complications, in remission  ← NEW in FY2026
```

> **2026 addition — `E11.A`:** *"Type 2 diabetes mellitus without complications in remission"* — confirmed valid for HIPAA transactions in the FY2026 ICD-10-CM release. This distinguishes patients who have achieved sustained normoglycemia (remission — e.g., post-bariatric surgery or sustained lifestyle intervention) from those still actively managed (E11.9).

> **Note on retinopathy codes:** The E11.3x subcategory is highly specific. Real-world MIMIC data often uses the unspecified form (`E11.319`) because the laterality (left/right/bilateral) was not recorded at the time of billing. ML models typically collapse these to the 3–4 character level (e.g., using CCS or grouping all `E11.3xx` together).

In [ ]:
# --- Type 1 Diabetes Mellitus (E10) codes ---
type1_codes = ["E10.9", "E10.65", "E10.319"]  # E10.3- = retinopathy

print("=" * 60)
print("TYPE 1 DIABETES MELLITUS (E10)")
print("=" * 60)
for code in type1_codes:
    if code in icd10cm:
        name = icd10cm.lookup(code, attribute="name")
        print(f"  {code:10s}: {name}")
    else:
        print(f"  {code:10s}: [not found in this ICD10CM version]")

In [ ]:
# --- Type 2 Diabetes Mellitus (E11) codes ---
type2_codes = [
    "E11.9",   # without complications
    "E11.65",  # with hyperglycemia
    "E11.22",  # with diabetic chronic kidney disease (CKD)
    "E11.42",  # with polyneuropathy
    "E11.319", # with unspecified diabetic retinopathy without macular edema
]

print("=" * 60)
print("TYPE 2 DIABETES MELLITUS (E11)")
print("=" * 60)
for code in type2_codes:
    if code in icd10cm:
        name = icd10cm.lookup(code, attribute="name")
        print(f"  {code:10s}: {name}")
    else:
        print(f"  {code:10s}: [not in this ICD10CM version]")

In [ ]:
# --- Other diabetes types ---
other_codes = [
    "E08.9",   # diabetes due to underlying condition, without complications
    "E09.9",   # drug/chemical-induced diabetes, without complications
    "E13.9",   # other specified diabetes mellitus, without complications
]

print("=" * 60)
print("OTHER DIABETES TYPES")
print("=" * 60)
for code in other_codes:
    if code in icd10cm:
        name = icd10cm.lookup(code, attribute="name")
        print(f"  {code:10s}: {name}")
    else:
        print(f"  {code:10s}: [not in this ICD10CM version]")

In [ ]:
# --- 2026 New Code: E11.A ---
# Confirmed valid for HIPAA transactions in FY2026 ICD-10-CM (verified against live 2026 dataset).
# Official description: "Type 2 diabetes mellitus without complications in remission"
#
# Clinical context:
#   E11.9 = T2DM without complications (still being managed / monitored)
#   E11.A = T2DM without complications IN REMISSION (achieved normoglycemia)
#
# Remission criteria (ADA 2021 Consensus): HbA1c < 6.5% for at least 3 months
# without the use of glucose-lowering pharmacotherapy.
# Common after bariatric surgery or significant sustained lifestyle intervention.

new_2026_code = "E11.A"
if new_2026_code in icd10cm:
    name = icd10cm.lookup(new_2026_code)
    print(f"[{new_2026_code}] {name}")
    print()
    print("Contrast with E11.9:")
    print(f"  [E11.9 ] {icd10cm.lookup('E11.9')}")
    print(f"  [E11.A ] {name}")
    print()
    print("Clinical significance:")
    print("  E11.9 → patient still has diabetes, actively managed")
    print("  E11.A → patient has achieved remission (ADA criteria: HbA1c < 6.5% for ≥3 months,")
    print("           no glucose-lowering medication)")
else:
    # Fallback if the PyHealth cache predates FY2026
    print("E11.A not yet in local ICD10CM cache.")
    print("Official 2026 description: 'Type 2 diabetes mellitus without complications in remission'")
    print("Update the cache with: InnerMap.load('ICD10CM', refresh_cache=True)")

---
## Part 3: Hierarchy Exploration

ICD codes form a tree where more specific codes are children of broader parent codes. PyHealth exposes this hierarchy via:
- `get_ancestors(code)` — returns parent codes, ordered from closest to farthest
- `get_descendants(code)` — returns child codes, ordered from closest to farthest

This hierarchy is stored internally as a **directed graph** using NetworkX.

In [ ]:
# --- Ancestor lookup: trace E11.22 up the hierarchy ---
# E11.22 = Type 2 diabetes mellitus with diabetic chronic kidney disease
code = "E11.22"
ancestors = icd10cm.get_ancestors(code)

print(f"Ancestors of {code} ({icd10cm.lookup(code) if code in icd10cm else 'T2DM with CKD'}):")
print(f"  [{code}] (starting code)")
for anc in ancestors:
    if anc in icd10cm:
        name = icd10cm.lookup(anc)
        print(f"  ↑ [{anc}] {name}")
    else:
        print(f"  ↑ [{anc}] (category node)")

In [ ]:
# --- Descendant lookup: find all Type 2 DM subtypes ---
# The live 2026 ICD-10-CM dataset has 87 billable E11 codes —
# reflecting the high granularity of modern diabetes coding (retinopathy
# laterality, ketoacidosis severity, complication type, etc.)

parent_code = "E11"
descendants = icd10cm.get_descendants(parent_code)

print(f"Descendants of {parent_code} (Type 2 Diabetes Mellitus):")
print(f"  Total subtypes in this ICD10CM version: {len(descendants)}")
print(f"  (FY2026 live dataset has 87 billable codes)")
print()

# Show the first 20 for readability — the full list is much longer
print("First 20 (sorted by code):")
for desc in sorted(descendants)[:20]:
    if desc in icd10cm:
        name = icd10cm.lookup(desc)
        print(f"  [{desc}] {name}")

In [ ]:
# --- Practical use: find all CKD-related diabetes codes ---
ckd_diabetes = [
    desc for desc in icd10cm.get_descendants("E11")
    if desc in icd10cm and "kidney" in icd10cm.lookup(desc).lower()
]

print("Type 2 DM codes involving kidney disease:")
for code in ckd_diabetes:
    print(f"  [{code}] {icd10cm.lookup(code)}")

In [ ]:
# --- Practical use: find all hyperglycemia-related diabetes codes ---
hyperglycemia_codes = [
    desc for desc in descendants
    if desc in icd10cm and "hyperglycemia" in icd10cm.lookup(desc).lower()
]

print("Hyperglycemia-related diabetes codes:")
for code in hyperglycemia_codes:
    print(f"  [{code}] {icd10cm.lookup(code)}")

---
## Part 4: Cross-System Code Translation with `CrossMap`

`CrossMap` translates codes from one vocabulary to another. This is essential when:
- Combining MIMIC-III (ICD-9) with MIMIC-IV (ICD-10) cohorts
- Reducing ICD-10's 70,000 codes to CCS's ~300 categories for modeling
- Mapping to clinical groupings used in published benchmarks

```python
CrossMap.load(source_vocabulary, target_vocabulary)
cm.map(source_code) -> List[str]  # returns a list (may be 1-to-many)
```

In [ ]:
# --- ICD-9-CM → ICD-10-CM ---
# MIMIC-III uses ICD-9; MIMIC-IV uses ICD-10.
# The GEM (General Equivalence Mappings) crosswalk handles this translation.

cm_9to10 = CrossMap.load("ICD9CM", "ICD10CM")

# ICD-9 diabetes codes (legacy MIMIC-III style)
icd9_diabetes = {
    "250.00": "Diabetes mellitus without mention of complication, type II",
    "250.10": "Diabetes with ketoacidosis, type II",
    "250.40": "Diabetes with renal manifestations, type II",
    "250.60": "Diabetes with neurological manifestations, type II",
}

print("ICD-9-CM → ICD-10-CM Diabetes Code Mapping:")
print()
for icd9_code, icd9_name in icd9_diabetes.items():
    icd10_codes = cm_9to10.map(icd9_code)
    print(f"  ICD-9  {icd9_code} — {icd9_name}")
    print(f"    → ICD-10: {icd10_codes}")
    for c in icd10_codes:
        if c in icd10cm:
            print(f"         [{c}] {icd10cm.lookup(c)}")
    print()

In [ ]:
# --- ICD-10-CM → CCS (Clinical Classifications Software) ---
# CCS groups ~70,000 ICD-10 codes into ~300 clinically meaningful categories.
# CCS category 49 = "Diabetes mellitus without complication"
# CCS category 50 = "Diabetes mellitus with complications"

cm_10toCCS = CrossMap.load("ICD10CM", "CCSCM")

print("ICD-10-CM → CCS Category Mapping:")
print()
type2_sample = ["E11.9", "E11.65", "E11.22", "E11.42", "E10.9"]
for code in type2_sample:
    ccs_cats = cm_10toCCS.map(code)
    icd_name = icd10cm.lookup(code) if code in icd10cm else "(not found)"
    print(f"  [{code}] {icd_name}")
    print(f"    → CCS: {ccs_cats}")
    print()

In [ ]:
# --- ICD-9-CM → CCS (direct, for MIMIC-III) ---
cm_9toCCS = CrossMap.load("ICD9CM", "CCSCM")

print("ICD-9-CM → CCS (useful for MIMIC-III preprocessing):")
print()
for icd9_code in ["250.00", "250.10", "250.40", "250.60"]:
    ccs_cats = cm_9toCCS.map(icd9_code)
    print(f"  {icd9_code} → CCS: {ccs_cats}")

---
## Part 5: Loading Other Code Systems

PyHealth supports many vocabularies beyond ICD-10-CM.

In [ ]:
# --- ICD-9-CM for legacy MIMIC-III data ---
icd9cm = InnerMap.load("ICD9CM")
icd9cm.stat()
print("Example lookup:", icd9cm.lookup("250.00"))
print()

In [ ]:
# --- ATC (Anatomical Therapeutic Chemical) for drug classification ---
# ATC hierarchy: Level1 (organ system) → Level2 (main group) → Level3 → Level4 → Level5 (substance)
# A10BA02 = Metformin
atc = InnerMap.load("ATC")
atc.stat()

metformin_code = "A10BA02"
if metformin_code in atc:
    print(f"ATC code {metformin_code}: {atc.lookup(metformin_code)}")
    ancestors_atc = atc.get_ancestors(metformin_code)
    print("Ancestors (drug class hierarchy):")
    for anc in ancestors_atc:
        if anc in atc:
            print(f"  ↑ [{anc}] {atc.lookup(anc)}")

---
## Part 6: Practical ML Preprocessing Patterns

Here are the most common patterns for using `medcode` when preparing datasets for ML.

In [ ]:
# Pattern 1: Filter codes to a clinical domain (e.g., all diabetes codes)
# Useful when restricting a cohort to patients with a specific condition.

all_diabetes_codes = set(icd10cm.get_descendants("E08")) | \
                     set(icd10cm.get_descendants("E09")) | \
                     set(icd10cm.get_descendants("E10")) | \
                     set(icd10cm.get_descendants("E11")) | \
                     set(icd10cm.get_descendants("E13"))
all_diabetes_codes |= {"E08", "E09", "E10", "E11", "E13"}  # include root codes

print(f"Total ICD-10-CM diabetes codes: {len(all_diabetes_codes)}")
print("Sample:", sorted(list(all_diabetes_codes))[:10])

In [ ]:
# Pattern 2: Translate a patient's ICD-9 code list to CCS before modeling
# This dramatically reduces vocabulary size.

def translate_codes_to_ccs(icd9_codes, crossmap):
    """Map a list of ICD-9 codes to CCS categories."""
    ccs_codes = []
    for code in icd9_codes:
        mapped = crossmap.map(code)
        ccs_codes.extend(mapped)
    return list(set(ccs_codes))  # deduplicate

# Example patient from MIMIC-III
patient_icd9_codes = ["250.00", "401.9", "428.0", "585.3"]
# = Type 2 DM, Essential hypertension, Heart failure, CKD stage 3

patient_ccs_codes = translate_codes_to_ccs(patient_icd9_codes, cm_9toCCS)
print("ICD-9 codes:", patient_icd9_codes)
print("CCS codes:  ", patient_ccs_codes)
print(f"Vocabulary reduction: {len(icd9cm.graph.nodes)} ICD-9 codes → ~300 CCS categories")

In [ ]:
# Pattern 3: Code validation — remove invalid/unknown codes before training
# Real EHR data often contains typos, deprecated codes, and free-text entries.

raw_codes_from_ehr = ["E11.9", "E11.99", "DIAB", "E11.65", "999.999", "E10.9"]
valid_codes   = [c for c in raw_codes_from_ehr if c in icd10cm]
invalid_codes = [c for c in raw_codes_from_ehr if c not in icd10cm]

print("Raw codes from EHR:",  raw_codes_from_ehr)
print("Valid ICD-10-CM codes:",   valid_codes)
print("Invalid / unknown codes:", invalid_codes)

---
## Summary

| Task | API |
|------|-----|
| Load code system | `icd10cm = InnerMap.load("ICD10CM")` |
| Look up a code | `icd10cm.lookup("E11.9")` |
| Check if code exists | `"E11.9" in icd10cm` |
| Print system stats | `icd10cm.stat()` |
| See available attributes | `icd10cm.available_attributes` |
| Get parent codes | `icd10cm.get_ancestors("E11.22")` |
| Get child codes | `icd10cm.get_descendants("E11")` |
| Translate between systems | `cm = CrossMap.load("ICD9CM", "ICD10CM"); cm.map("250.00")` |
| Reduce to CCS groups | `cm = CrossMap.load("ICD10CM", "CCSCM"); cm.map("E11.9")` |

### Supported vocabularies

| Vocabulary name | Description |
|-----------------|-------------|
| `ICD9CM` | ICD-9-CM diagnosis codes (MIMIC-III) |
| `ICD10CM` | ICD-10-CM diagnosis codes (MIMIC-IV, current US standard) |
| `ICD9PROC` | ICD-9 procedure codes |
| `ICD10PROC` | ICD-10-PCS procedure codes |
| `ATC` | Anatomical Therapeutic Chemical drug classification |
| `NDC` | National Drug Code (US drug packaging) |
| `RxNorm` | RxNorm drug concepts |
| `CCSCM` | CCS diagnosis categories |
| `CCSPROC` | CCS procedure categories |
| `UMLS` | Unified Medical Language System concepts |